In [1]:
!pip install pandas numpy matplotlib pyarrow scikit-learn tkan "jax[cuda12]" keras_sig tkan sig_rnn --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 20.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 32.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 36.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 31.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 38.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 41.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 KB 36.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.0/508.0 KB 39.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 KB 28.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 43.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Exploring Signature-Based Gating in LSTM and GRU Networks

This notebook investigates the integration of path signatures into classical recurrent neural network architectures, focusing on their application in gating mechanisms. We explore two different approaches:

1. **Signature-Based LSTM**: A modified LSTM where the forget gate is controlled by path signatures instead of traditional linear transformations, while maintaining the standard computation for input and output gates.

2. **Signature-Based GRU**: An adaptation of the Gated Recurrent Unit where the reset gate computation is enhanced with path signatures, preserving the original update gate mechanics.

## Motivation

Traditional LSTM and GRU architectures use linear transformations followed by nonlinear activations for their gating mechanisms. However, path signatures provide a rich, hierarchical representation of sequential data that could potentially offer more sophisticated control over information flow in these networks. By incorporating signatures into specific gating mechanisms, we aim to:

- Capture higher-order temporal patterns in the input sequences
- Provide better control over long-term dependencies through signature-based forget/reset mechanisms
- Maintain the proven effectiveness of standard gates while enhancing critical gates with signature computations

## Implementation Details

The notebook implements these architectures using Keras 3's backend-agnostic framework, ensuring compatibility across TensorFlow, JAX, and PyTorch backends. Each implementation:
- Uses streaming signatures to maintain temporal coherence
- Normalizes signature values by sequence length
- Maintains the original architecture's core functionality while enhancing specific gates
- Computes signatures at the RNN level to optimize computational efficiency

## Usage

The implementations can be used as drop-in replacements for standard LSTM and GRU layers in any Keras model. They accept the same input formats and provide the same output structures as their traditional counterparts.

---

*Note: This research explores the potential benefits of combining classical RNN architectures with modern signature methods for enhanced temporal data processing.*

In [2]:
import os
BACKEND = 'jax' # You can use any backend here 
os.environ['KERAS_BACKEND'] = BACKEND

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Input, Flatten, GRU

from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error

from tkan import TKAN
from keras_sig import SigLayer

from sig_rnn import SignatureGRU,SignatureLSTM

import time

keras.utils.set_random_seed(1) 

N_MAX_EPOCHS = 1000
BATCH_SIZE = 128
early_stopping_callback = lambda : keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0.00001,
    patience=10,
    mode="min",
    restore_best_weights=True,
    start_from_epoch=6,
)
lr_callback = lambda : keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.25,
    patience=5,
    mode="min",
    min_delta=0.00001,
    min_lr=0.000025,
    verbose=0,
)
callbacks = lambda : [early_stopping_callback(), lr_callback(), keras.callbacks.TerminateOnNaN()]


# Data

In [3]:
df = pd.read_parquet('data.parquet')
df = df[(df.index >= pd.Timestamp('2020-01-01')) & (df.index < pd.Timestamp('2023-01-01'))]
assets = ['BTC', 'ETH', 'ADA', 'XMR', 'EOS', 'MATIC', 'TRX', 'FTM', 'BNB', 'XLM', 'ENJ', 'CHZ', 'BUSD', 'ATOM', 'LINK', 'ETC', 'XRP', 'BCH', 'LTC']
df = df[[c for c in df.columns if 'quote asset volume' in c and any(asset in c for asset in assets)]]
df.columns = [c.replace(' quote asset volume', '') for c in df.columns]
display(df)

,BTC,ADA,XMR,EOS,CHZ,MATIC,TRX,ENJ,FTM,BNB,XLM,BUSD,ATOM,LTC,LINK,ETC,ETH,XRP,BCH
group,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,3.675857e+06,38189.176211,4.539598e+04,94778.577031,817.146319,31003.791035,481993.354990,15241.945783,1165.788613,8.498617e+05,9460.819556,1.352376e+04,31986.972694,1.165827e+05,24281.170262,56488.402352,1.000930e+06,2.579254e+05,178258.749391
2020-01-01 01:00:00,6.365953e+06,51357.010954,3.348395e+04,593292.135445,886.460339,84465.335718,533668.554562,11896.843688,413.844612,7.405759e+05,37141.909518,2.531605e+04,81777.666046,2.830715e+05,51190.975142,182102.074213,1.474278e+06,4.520609e+05,615321.025242
2020-01-01 02:00:00,4.736719e+06,36164.263914,1.573255e+04,266732.556000,1819.795050,113379.718506,387049.986770,30109.770521,3559.965968,1.039091e+06,16878.822627,1.390886e+04,195731.175551,2.402871e+05,28721.756184,134063.422732,9.940256e+05,4.414948e+05,221535.645771
2020-01-01 03:00:00,5.667367e+06,24449.953815,2.575105e+04,124516.579473,2979.655803,41771.707995,450772.139235,6732.833578,4076.415482,4.975018e+05,9049.223394,2.251969e+04,120113.343316,1.613043e+05,29596.222534,131094.172168,6.473610e+05,1.886061e+05,397185.950571
2020-01-01 04:00:00,3.379094e+06,44502.669843,6.295563e+04,421819.671410,1023.388675,22254.756114,284788.973752,846.938455,633.367505,4.751285e+05,7254.260203,1.122460e+04,19989.169106,2.214516e+05,54514.370016,134937.122201,4.430067e+05,2.279373e+05,316499.137509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-31 19:00:00,6.704605e+07,581680.400510,3.873989e+05,48359.865300,199491.822800,890911.573610,225136.420055,40281.859330,159553.944500,9.889098e+05,39230.588600,6.560756e+06,180809.784710,9.964355e+05,190664.976300,181340.756100,7.738029e+06,1.413563e+06,35409.149500
2022-12-31 20:00:00,4.344849e+07,323561.762270,1.379392e+05,37858.704700,173057.240300,333511.762200,157069.026827,42228.830930,270251.374500,6.032059e+05,52964.531800,7.255324e+06,276013.421720,1.173164e+06,265727.950340,90513.087600,4.278879e+06,1.113527e+06,42674.516600
2022-12-31 21:00:00,5.992803e+07,455185.698060,2.445869e+05,79538.050600,107544.609700,525037.759990,180404.744820,27446.620810,198885.610000,1.386864e+06,44485.594800,8.712142e+06,476151.071190,6.820723e+05,265687.852060,85399.066100,4.643401e+06,1.373231e+06,38027.858800


In [4]:
class MinMaxScaler:
    def __init__(self, feature_axis=None, minmax_range=(0, 1)):
        """
        Initialize the MinMaxScaler.
        Args:
        feature_axis (int, optional): The axis that represents the feature dimension if applicable.
                                      Use only for 3D data to specify which axis is the feature axis.
                                      Default is None, automatically managed based on data dimensions.
        """
        self.feature_axis = feature_axis
        self.min_ = None
        self.max_ = None
        self.scale_ = None
        self.minmax_range = minmax_range # Default range for scaling (min, max)

    def fit(self, X):
        """
        Fit the scaler to the data based on its dimensionality.
        Args:
        X (np.array): The data to fit the scaler on.
        """
        if X.ndim == 3 and self.feature_axis is not None:  # 3D data
            axis = tuple(i for i in range(X.ndim) if i != self.feature_axis)
            self.min_ = np.min(X, axis=axis)
            self.max_ = np.max(X, axis=axis)
        elif X.ndim == 2:  # 2D data
            self.min_ = np.min(X, axis=0)
            self.max_ = np.max(X, axis=0)
        elif X.ndim == 1:  # 1D data
            self.min_ = np.min(X)
            self.max_ = np.max(X)
        else:
            raise ValueError("Data must be 1D, 2D, or 3D.")

        self.scale_ = self.max_ - self.min_
        return self

    def transform(self, X):
        """
        Transform the data using the fitted scaler.
        Args:
        X (np.array): The data to transform.
        Returns:
        np.array: The scaled data.
        """
        X_scaled = (X - self.min_) / self.scale_
        X_scaled = X_scaled * (self.minmax_range[1] - self.minmax_range[0]) + self.minmax_range[0]
        return X_scaled

    def fit_transform(self, X):
        """
        Fit to data, then transform it.
        Args:
        X (np.array): The data to fit and transform.
        Returns:
        np.array: The scaled data.
        """
        return self.fit(X).transform(X)

    def inverse_transform(self, X_scaled):
        """
        Inverse transform the scaled data to original data.
        Args:
        X_scaled (np.array): The scaled data to inverse transform.
        Returns:
        np.array: The original data scale.
        """
        X = (X_scaled - self.minmax_range[0]) / (self.minmax_range[1] - self.minmax_range[0])
        X = X * self.scale_ + self.min_
        return X

def generate_data(df, sequence_length, n_ahead = 1):
    #Case without known inputs
    scaler_df = df.copy().shift(n_ahead).rolling(24 * 14).median()
    tmp_df = df.copy() / scaler_df
    tmp_df = tmp_df.iloc[24 * 14 + n_ahead:].fillna(0.)
    scaler_df = scaler_df.iloc[24 * 14 + n_ahead:].fillna(0.)
    def prepare_sequences(df, scaler_df, n_history, n_future):
        X, y, y_scaler = [], [], []
        num_features = df.shape[1]
        
        # Iterate through the DataFrame to create sequences
        for i in range(n_history, len(df) - n_future + 1):
            # Extract the sequence of past observations
            X.append(df.iloc[i - n_history:i].values)
            # Extract the future values of the first column
            y.append(df.iloc[i:i + n_future,0:1].values)
            y_scaler.append(scaler_df.iloc[i:i + n_future,0:1].values)
        
        X, y, y_scaler = np.array(X), np.array(y), np.array(y_scaler)
        return X, y, y_scaler
    
    # Prepare sequences
    X, y, y_scaler = prepare_sequences(tmp_df, scaler_df, sequence_length, n_ahead)
    
    # Split the dataset into training and testing sets
    train_test_separation = int(len(X) * 0.8)
    X_train_unscaled, X_test_unscaled = X[:train_test_separation], X[train_test_separation:]
    y_train_unscaled, y_test_unscaled = y[:train_test_separation], y[train_test_separation:]
    y_scaler_train, y_scaler_test = y_scaler[:train_test_separation], y_scaler[train_test_separation:]
    
    # Generate the data
    X_scaler = MinMaxScaler(feature_axis=2)
    X_train = X_scaler.fit_transform(X_train_unscaled)
    X_test = X_scaler.transform(X_test_unscaled)
    
    y_scaler = MinMaxScaler(feature_axis=2)
    y_train = y_scaler.fit_transform(y_train_unscaled)
    y_test = y_scaler.transform(y_test_unscaled)
    
    y_train = y_train.reshape(y_train.shape[0], -1) 
    y_test = y_test.reshape(y_test.shape[0], -1)
    return X_scaler, X_train, X_test, X_train_unscaled, X_test_unscaled, y_scaler, y_train, y_test, y_train_unscaled, y_test_unscaled, y_scaler_train, y_scaler_test



In [5]:
n_aheads = [1, 9, 15]
models = [
    "SignatureLSTM-2-2",
    "SignatureGRU-2-2",
    "SignatureLSTM-3-2",
    "SignatureGRU-3-2",
    "SignatureLSTM-3-3",
    "SignatureGRU-3-3",
    "SignatureLSTM-2_10-2",
    "SignatureGRU-2_10-2",
    "SignatureLSTM-3-3-3",
    "SignatureGRU-3-3-3",
    "SignatureLSTM-4-4",
    "SignatureGRU-4-4",
    "SignatureLSTM-3-3-flatten",
    "SignatureGRU-3-3-flatten",
    "GRU",
    "LSTM",
    "GRU-3",
    "LSTM-3",
    "GRU-flatten",
    "LSTM-flatten",
 ]

results = {model: {n_ahead: [] for n_ahead in n_aheads} for model in models}
results_rmse = {model: {n_ahead: [] for n_ahead in n_aheads} for model in models}
time_results = {model: {n_ahead: [] for n_ahead in n_aheads} for model in models}
for n_ahead in n_aheads:
    sequence_length = max(45, 5 * n_ahead)
    for run in range(5):
        X_scaler, X_train, X_test, X_train_unscaled, X_test_unscaled, y_scaler, y_train, y_test, y_train_unscaled, y_test_unscaled, y_scaler_train, y_scaler_test = generate_data(df, sequence_length, n_ahead)
        
        for model_id in models:

            if model_id == 'LSTM':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    LSTM(100, return_sequences=True),
                    LSTM(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'GRU':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    GRU(100, return_sequences=True),
                    GRU(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-2-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, return_sequences=True),
                    SignatureLSTM(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-2-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, return_sequences=True),
                    SignatureGRU(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-3-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    SignatureLSTM(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-3-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    SignatureGRU(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-3-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    SignatureLSTM(100, signature_depth=3, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-3-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    SignatureGRU(100, signature_depth=3, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-2_10-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_input_size=10, return_sequences=True),
                    SignatureLSTM(100, signature_input_size=10, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-2_10-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_input_size=10, return_sequences=True),
                    SignatureGRU(100, signature_input_size=10, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'LSTM-flatten':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    LSTM(100, return_sequences=True),
                    LSTM(100, return_sequences=True),
                    Flatten(),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'GRU-flatten':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    GRU(100, return_sequences=True),
                    GRU(100, return_sequences=True),
                    Flatten(),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'LSTM-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    LSTM(100, return_sequences=True),
                    LSTM(100, return_sequences=True),
                    LSTM(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'GRU-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    GRU(100, return_sequences=True),
                    GRU(100, return_sequences=True),
                    GRU(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-3-3-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    SignatureLSTM(100, signature_depth=3, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-3-3-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    SignatureGRU(100, signature_depth=3, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-4-4':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_depth=4, return_sequences=True),
                    SignatureLSTM(100, signature_depth=4, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-4-4':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_depth=4, return_sequences=True),
                    SignatureGRU(100, signature_depth=4, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-3-3-flatten':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    Flatten(),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-3-3-flatten':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    Flatten(),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            else:
                raise ValueError

            optimizer = keras.optimizers.Adam(0.001)
            model.compile(optimizer=optimizer, loss='mean_squared_error', jit_compile=True)
            if run==0:
                model.summary()
                
            # Fit the model
            start_time = time.time()
            history = model.fit(X_train, y_train, batch_size=BATCH_SIZE, epochs=N_MAX_EPOCHS, validation_split=0.2, callbacks=callbacks(), shuffle=True, verbose = False)
            end_time = time.time()
            time_results[model_id][n_ahead].append(end_time - start_time)
            # Evaluate the model on the test set
            preds = model.predict(X_test, verbose=False)
            r2 = r2_score(y_true=y_test, y_pred=preds)
            print(model_id, end_time - start_time, r2)
            rmse = root_mean_squared_error(y_true=y_test, y_pred=preds)
            results[model_id][n_ahead].append(r2)
            results_rmse[model_id][n_ahead].append(rmse)
    
            del model
            del optimizer
                

print('R2 scores')
print('Means:')
df_mean_r2 = pd.DataFrame({model_id: {n_ahead: np.mean(results[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in results.keys()})
df_mean_r2.to_csv('volume_mean_r2.csv')
display(df_mean_r2)
df_mean_rmse = pd.DataFrame({model_id: {n_ahead: np.mean(results_rmse[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in results_rmse.keys()})
df_mean_rmse.to_csv('volume_mean_rmse.csv')
display(df_mean_rmse)
print('Std:')
df_std_r2 = pd.DataFrame({model_id: {n_ahead: np.std(results[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in results.keys()})
df_std_r2.to_csv('volume_std_r2.csv')
display(df_std_r2)
df_std_rmse = pd.DataFrame({model_id: {n_ahead: np.std(results_rmse[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in results_rmse.keys()})
df_std_rmse.to_csv('volume_std_rmse.csv')
display(df_std_rmse)
print('Training Times')
df_mean_time = pd.DataFrame({model_id: {n_ahead: np.mean(time_results[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in time_results.keys()})
df_mean_time.to_csv('volume_mean_time.csv')
display(df_mean_time)
df_std_time = pd.DataFrame({model_id: {n_ahead: np.std(time_results[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in time_results.keys()})
df_std_time.to_csv('volume_std_time.csv')
display(df_std_time)

Model: "SignatureLSTM-2-2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ signature_lstm (SignatureLSTM)  │ (None, 45, 100)        │        39,195 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ signature_lstm_1                │ (None, 100)            │        63,900 │
│ (SignatureLSTM)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,196 (403.11 KB)

 Trainable params: 103,196 (403.11 KB)

 Non-trainable params: 0 (0.00 B)

...

,SignatureLSTM-2-2,SignatureGRU-2-2,SignatureLSTM-3-2,SignatureGRU-3-2,SignatureLSTM-3-3,SignatureGRU-3-3,SignatureLSTM-2_10-2,SignatureGRU-2_10-2,SignatureLSTM-3-3-3,SignatureGRU-3-3-3,SignatureLSTM-4-4,SignatureGRU-4-4,SignatureLSTM-3-3-flatten,SignatureGRU-3-3-flatten,GRU,LSTM,GRU-3,LSTM-3,GRU-flatten,LSTM-flatten
1,0.017137,0.007043,0.005855,0.003245,0.013586,0.001924,0.013236,0.003651,0.003630,0.004506,0.010070,0.004756,0.002765,0.005555,0.001772,0.006063,0.007904,0.021212,0.002615,0.007131
9,0.201650,0.009848,0.052136,0.007674,0.082754,0.007757,0.120547,0.011442,0.195712,0.008025,0.094394,0.004078,0.036824,0.008835,0.022377,0.128758,0.053588,0.122811,0.010675,0.226580
15,0.094039,0.004549,0.116707,0.007957,0.146341,0.006220,0.086677,0.003731,0.072477,0.006531,0.072624,0.007596,0.046918,0.012335,0.041139,0.136899,0.063434,0.046717,0.004123,0.482613


,SignatureLSTM-2-2,SignatureGRU-2-2,SignatureLSTM-3-2,SignatureGRU-3-2,SignatureLSTM-3-3,SignatureGRU-3-3,SignatureLSTM-2_10-2,SignatureGRU-2_10-2,SignatureLSTM-3-3-3,SignatureGRU-3-3-3,SignatureLSTM-4-4,SignatureGRU-4-4,SignatureLSTM-3-3-flatten,SignatureGRU-3-3-flatten,GRU,LSTM,GRU-3,LSTM-3,GRU-flatten,LSTM-flatten
1,0.000779,0.000322,0.000267,0.000149,0.000618,0.000088,0.000598,0.000168,0.000166,0.000205,0.000458,0.000218,0.000127,0.000253,0.000081,0.000274,0.000356,0.000936,0.000118,0.000322
9,0.006356,0.000367,0.001794,0.000289,0.002674,0.000281,0.003834,0.000420,0.005817,0.000297,0.003078,0.000148,0.001331,0.000324,0.000817,0.004228,0.001883,0.004107,0.000387,0.006687
15,0.003097,0.000165,0.003854,0.000289,0.004757,0.000242,0.002840,0.000140,0.002380,0.000236,0.002462,0.000279,0.001660,0.000442,0.001483,0.004287,0.002246,0.001541,0.000150,0.013320


Training Times


,SignatureLSTM-2-2,SignatureGRU-2-2,SignatureLSTM-3-2,SignatureGRU-3-2,SignatureLSTM-3-3,SignatureGRU-3-3,SignatureLSTM-2_10-2,SignatureGRU-2_10-2,SignatureLSTM-3-3-3,SignatureGRU-3-3-3,SignatureLSTM-4-4,SignatureGRU-4-4,SignatureLSTM-3-3-flatten,SignatureGRU-3-3-flatten,GRU,LSTM,GRU-3,LSTM-3,GRU-flatten,LSTM-flatten
1,82.656979,41.744139,85.626511,47.768060,86.771965,43.893235,89.371870,46.291192,122.443510,66.963048,97.229167,51.152382,128.537148,48.967095,19.077594,14.706821,28.582612,22.822792,18.777823,18.310477
9,82.514562,47.771547,86.710444,51.977176,87.781003,46.830489,87.388115,47.304620,126.402231,62.162630,93.566093,53.949484,87.423395,45.549802,19.192821,18.458972,26.554420,24.007343,20.804365,17.698956
15,154.055377,50.309314,159.632405,58.418006,160.486684,52.999086,155.903946,52.020234,215.357774,78.302664,174.183531,65.139287,162.894418,59.602691,25.040493,21.945757,41.775051,30.489373,28.865783,25.375856


,SignatureLSTM-2-2,SignatureGRU-2-2,SignatureLSTM-3-2,SignatureGRU-3-2,SignatureLSTM-3-3,SignatureGRU-3-3,SignatureLSTM-2_10-2,SignatureGRU-2_10-2,SignatureLSTM-3-3-3,SignatureGRU-3-3-3,SignatureLSTM-4-4,SignatureGRU-4-4,SignatureLSTM-3-3-flatten,SignatureGRU-3-3-flatten,GRU,LSTM,GRU-3,LSTM-3,GRU-flatten,LSTM-flatten
1,3.126633,2.326170,4.308339,3.692054,4.252355,1.548704,4.234455,2.878468,3.294972,3.521717,3.624107,2.072887,2.312145,3.682688,2.403916,0.904345,5.240006,2.129134,1.091168,2.609470
9,6.658195,1.693063,3.300962,2.515764,4.954757,2.992510,3.796205,1.659722,2.740668,3.675876,3.798537,3.098326,5.263704,2.236229,2.142387,1.248940,3.025918,1.675379,0.954727,2.013436
15,1.410502,1.355652,5.208323,2.913191,2.199755,3.093837,4.949384,3.076429,6.966844,6.161856,5.291203,3.574824,3.180684,3.552262,3.693775,1.806929,6.677514,0.727149,2.726082,3.197511
